# Visual Diagnostics for Stationarity

**Week 2 — Stationarity | Notebook 1 of 3**

Before running any statistical test, we start with **visual inspection**. A stationary series has:
- Constant mean over time (no trend)
- Constant variance over time (no fanning/narrowing)

Rolling mean and rolling variance plots are the primary visual tool for detecting violations.

This notebook compares a clearly **non-stationary** series (Airline Passengers) with a roughly **stationary** one (Sunspots) to build intuition for what stationarity looks like.

In [ ]:
import sys
sys.path.insert(0, "../..")

import matplotlib.pyplot as plt
import seaborn as sns

from week__01__fundamentals.ts_loader import TimeSeriesLoader
from week__02__stationarity.stationarity_tests import StationarityTester

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 4)

## 1. Strict vs Weak (Covariance) Stationarity

Before looking at any data, we need to understand **what stationarity actually means**. There are two definitions:

**Strict stationarity:**
The *entire* joint probability distribution of (yₜ, yₜ₊₁, ..., yₜ₊ₖ) is identical for all t.
Every statistical property — mean, variance, skewness, kurtosis, all higher moments — is time-invariant.
This is extremely restrictive and almost impossible to verify in practice.

**Weak (covariance) stationarity:**
Only the *first two moments* are time-invariant:
1. E[yₜ] = μ (constant mean)
2. Var(yₜ) = σ² (constant variance)
3. Cov(yₜ, yₜ₊ₖ) = γ(k) (autocovariance depends only on lag k, not on t)

**In practice, "stationary" almost always means weak stationarity.** This is what ADF, KPSS, and ARIMA care about.

The distinction matters because a series can be weakly stationary without being strictly stationary
(e.g., if its higher moments change over time), and a strictly stationary series with infinite variance
(e.g., Cauchy draws) is NOT weakly stationary.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)
n = 1000
dates = pd.date_range("2000-01-01", periods=n, freq="D")

# --- Example 1: Weakly stationary (Gaussian white noise) ---
# Constant mean (0), constant variance (1), zero autocovariance at all lags.
# Also strictly stationary because Gaussian ⇒ distribution fully determined by first two moments.
weak_stationary = pd.Series(rng.standard_normal(n), index=dates, name="Gaussian White Noise")

# --- Example 2: Weakly stationary but NOT strictly stationary ---
# Trick: use a process whose first two moments are constant but higher moments change.
# Take a chi-squared(1) process but shift and scale to have constant mean=0, var=1,
# then alternate its skewness sign over time.
half = n // 2
chi2_pos = (rng.chisquare(df=1, size=half) - 1) / np.sqrt(2)   # mean=0, var=1, positive skew
chi2_neg = -((rng.chisquare(df=1, size=half) - 1) / np.sqrt(2)) # mean=0, var=1, NEGATIVE skew
weak_not_strict = pd.Series(
    np.concatenate([chi2_pos, chi2_neg]), index=dates,
    name="Weak-only (skewness flips)"
)

# --- Example 3: Not even weakly stationary (random walk) ---
random_walk = pd.Series(
    np.cumsum(rng.standard_normal(n)), index=dates, name="Random Walk"
)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, (series, subtitle) in zip(axes, [
    (weak_stationary, "Gaussian WN — strictly AND weakly stationary"),
    (weak_not_strict, "Skewness flips at midpoint — weakly stationary, NOT strictly"),
    (random_walk, "Random walk — NOT stationary at all (variance grows with t)"),
]):
    ax.plot(series, linewidth=0.5, alpha=0.8)
    ax.axhline(y=series.mean(), color="red", linestyle="--", linewidth=1.5, label="Overall mean")
    ax.set_title(subtitle)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

# Verify: first two moments are constant across halves for the "weak-only" series
for label, s in [("Gaussian WN", weak_stationary), ("Weak-only", weak_not_strict), ("Random Walk", random_walk)]:
    first, second = s.iloc[:half], s.iloc[half:]
    print(f"{label:15s}  mean₁={first.mean():+.3f}  mean₂={second.mean():+.3f}  "
          f"var₁={first.var():.3f}  var₂={second.var():.3f}  "
          f"skew₁={first.skew():+.3f}  skew₂={second.skew():+.3f}")

**Interpretation of the output table:**
- **Gaussian WN**: mean, variance, AND skewness are stable across halves → both strictly and weakly stationary.
- **Weak-only**: mean and variance are near-identical across halves (weak stationarity holds), but **skewness flips sign** — the distribution shape changed, so strict stationarity is violated.
- **Random Walk**: variance in the second half is much larger than the first — even weak stationarity fails.

**Takeaway for time series modelling:** ADF and KPSS test for *weak* stationarity (constant mean and variance). They cannot detect violations of strict stationarity. For ARIMA, weak stationarity is sufficient.

## 2. Load Datasets

In [ ]:
loader_airline = TimeSeriesLoader(name="AirPassengers")
airline, airline_summary = loader_airline.from_statsmodels("airline")
print(airline_summary)

loader_sunspots = TimeSeriesLoader(name="Sunspots")
sunspots, sunspots_summary = loader_sunspots.from_statsmodels("sunspots")
print(sunspots_summary)

## 3. Raw Series Plots

First, just look at the raw data. What do we see?
- **Airline Passengers**: clear upward trend + seasonal amplitude that grows with level → non-stationary
- **Sunspots**: no persistent trend, roughly constant amplitude → plausibly stationary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(airline, color="steelblue")
axes[0].set_title("Airline Passengers — clearly non-stationary")
axes[0].set_ylabel("Passengers")

axes[1].plot(sunspots, color="coral")
axes[1].set_title("Sunspots — roughly stationary")
axes[1].set_ylabel("Sunspot count")

plt.tight_layout()
plt.show()

## 4. Rolling Mean and Rolling Variance

The rolling window approach formalises the visual test:
- **Rolling mean**: if it trends up/down, the series has a non-constant mean → non-stationary
- **Rolling variance**: if it grows over time, the series has heteroscedastic variance → non-stationary

We use the `StationarityTester.rolling_diagnostics()` static method.

In [ ]:
def plot_rolling_diagnostics(series: "pd.Series", window: int, title: str) -> None:
    """Plot raw series with overlaid rolling mean and a separate rolling variance panel."""
    diag = StationarityTester.rolling_diagnostics(series, window=window)

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # Top: raw + rolling mean
    axes[0].plot(series, alpha=0.5, label="Raw")
    axes[0].plot(diag["rolling_mean"], color="red", linewidth=2, label=f"Rolling Mean (w={window})")
    axes[0].set_title(f"{title} — Rolling Mean")
    axes[0].legend()

    # Bottom: rolling variance
    axes[1].plot(diag["rolling_var"], color="darkorange", linewidth=2)
    axes[1].set_title(f"{title} — Rolling Variance")
    axes[1].set_ylabel("Variance")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rolling_diagnostics(airline, window=12, title="Airline Passengers")

**Interpretation — Airline Passengers:**
- Rolling mean is clearly trending upward — the mean is not constant over time.
- Rolling variance increases dramatically — the seasonal swings get wider as the level rises.
- Both violations confirm: this series is **non-stationary** and needs both a variance-stabilising transform (log) and differencing.

In [ ]:
plot_rolling_diagnostics(sunspots, window=11, title="Sunspots")

**Interpretation — Sunspots:**
- Rolling mean oscillates around a roughly constant level — no persistent trend.
- Rolling variance fluctuates but does not systematically increase or decrease.
- Visual verdict: this series is **plausibly stationary** (though the ADF/KPSS tests in Notebook 2 will confirm formally).

## 5. Key Takeaway

Visual diagnostics are the **first step**, not a replacement for formal tests. They help us:
1. Decide whether a variance-stabilising transform is needed (growing variance → yes)
2. Build intuition for what the ADF/KPSS tests should confirm
3. Catch obvious problems (structural breaks, outliers) that tests may not flag

**Next:** Notebook 2 formalises these observations with ADF and KPSS tests.